In [2]:
import numpy as np
import math

L, d_k, d_v =4,8,8
q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

# Self-Attention (Scaled Dot-Product Attention)

Minh hoạ cách self-attention hoạt động trên một chuỗi ngắn (L=4 token), với Q, K, V được sinh ngẫu nhiên, kích thước `d_k = d_v = 8`.

In [3]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)


Q
 [[-1.57022193  1.36077008 -0.89528795  0.4129017   0.25335323  0.02778555
   1.62836619  0.88931179]
 [-0.58418328  0.5129718  -1.53355145  0.69463447 -1.30660233 -0.06647296
  -0.93841362  2.68778354]
 [-1.64184951  1.92822597 -0.11088405  0.43824264 -0.00870615  0.03803076
   0.42473608 -0.04186264]
 [-2.07997765  2.55894099 -2.28013236  1.4892446  -0.1632833  -0.47578329
   2.11477084 -0.70708352]]
K
 [[ 0.67182034  1.28598954 -1.05083004  0.85078559 -0.99111404 -0.47746124
   0.36717127  0.23699898]
 [-0.71446852  0.04792653  1.41017261  0.99519711 -1.48809583 -1.49773556
   0.99982822 -0.44236456]
 [-0.89614036 -2.0150702  -2.30332924 -0.94571842 -0.68161453  1.31809401
   0.49706184  1.40281078]
 [ 0.91711568  1.5727808  -0.27434931 -0.31215048 -0.58853803 -0.09901833
   1.04709807  1.3507955 ]]
V
 [[ 1.89729554e+00  8.46678539e-01  4.97330176e-01  1.25709236e+00
  -4.21785213e-01  7.19673829e-02 -1.12461151e+00 -3.56175090e-01]
 [-1.99781611e+00 -2.43499718e-01  1.72230363e+0


$$
\text{self attention} = softmax\left(\frac{Q \cdot K^T}{\sqrt{d_k}} + M\right)V
$$




In [4]:
#nhân ma trận Q . KT
np.matmul(q, k.T)

array([[ 2.53140215,  1.15155633,  2.25761575,  3.57133023],
       [ 4.08887071, -1.1126327 ,  6.47216494,  3.89852889],
       [ 2.00252139,  1.94441235, -2.36477765,  1.81008479],
       [ 6.55436495,  3.25820645,  0.09443235,  3.68021439]])

Kết quả `Q . K^T` là ma trận điểm tương đồng (raw attention score) giữa từng cặp token — phần tử (i, j) thể hiện mức độ "liên quan" giữa token i và token j, chưa được chuẩn hoá.

In [5]:
#tại sao lại cần phải chia cho sqrt(d_k) khi tính attention score?
q.var(), k.var(),np.matmul(q, k.T).var()

(np.float64(1.6274502777150532),
 np.float64(1.1475913463558465),
 np.float64(5.3265234003870106))

In [6]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), scaled.var()

(np.float64(1.6274502777150532),
 np.float64(1.1475913463558465),
 np.float64(0.6658154250483763))

Phương sai của `Q . K^T` (~5.3) lớn hơn nhiều so với Q, K riêng lẻ (~1.6, ~1.1). Nếu đưa thẳng giá trị lớn này vào softmax, hàm sẽ bị bão hoà (gradient gần như bằng 0 ở vùng bão hoà). Chia cho `sqrt(d_k)` kéo phương sai về gần mức ban đầu (~0.67), giúp huấn luyện ổn định hơn.

In [7]:
scaled

array([[ 0.89498581,  0.40713664,  0.7981877 ,  1.26265591],
       [ 1.4456341 , -0.39337506,  2.28825586,  1.37833811],
       [ 0.70799823,  0.68745358, -0.83607516,  0.63996162],
       [ 2.31731795,  1.15194994,  0.03338688,  1.30115228]])

Masking

In [8]:
mask = np.tril(np.ones((L, L)))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

Mask tam giác dưới (`np.tril`) dùng cho self-attention có tính nhân quả (causal), như trong decoder: token ở vị trí i chỉ được phép "nhìn thấy" các token từ 0..i, không được nhìn về tương lai.

In [9]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0

In [10]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

Vị trí bị che (tương lai) được gán `-inf`; vị trí được phép nhìn thấy gán `0`. Khi cộng mask vào scaled score rồi qua softmax, các vị trí `-inf` sẽ có xác suất chú ý ≈ 0.

In [11]:
scaled + mask

array([[ 0.89498581,        -inf,        -inf,        -inf],
       [ 1.4456341 , -0.39337506,        -inf,        -inf],
       [ 0.70799823,  0.68745358, -0.83607516,        -inf],
       [ 2.31731795,  1.15194994,  0.03338688,  1.30115228]])

$$
\text{softmax} = \frac{e^{x_i}}{\sum_j e^{x_j}}
$$

In [52]:
def softmax(x, axis=-1):
    x = np.exp(x)
    return x / np.sum(x, axis=axis, keepdims=True)

In [13]:
attention = softmax(scaled + mask)
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.86283148, 0.13716852, 0.        , 0.        ],
       [0.45596006, 0.44668809, 0.09735186, 0.        ],
       [0.56316738, 0.17560003, 0.05737713, 0.20385546]])

Gộp toàn bộ các bước ở trên (QK^T → chia scale → cộng mask → softmax → nhân với V) thành một hàm `scaled_dot_product_attention` để tái sử dụng.

In [55]:
def softmax(x, axis=-1):
    x = np.exp(x)
    return x / np.sum(x, axis=axis, keepdims=True)

def scaled_dot_product_attention(q, k, v, mask=None) :
    d_k = q.shape[-1]
    scaled = np.matmul(q, np.swapaxes(k, -2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scaled = scaled + mask
    attention = softmax(scaled, axis=-1)   # nhớ chỉ định axis=-1
    out = np.matmul(attention, v)
    return out, attention

Kiểm tra lại hàm với cùng Q, K, V, mask như trên — `values` là output cuối (mỗi token giờ là trung bình có trọng số của các V theo attention weight), `attention` là ma trận trọng số dùng để kiểm tra lại.

In [56]:
values,attention = scaled_dot_product_attention(q, k, v, mask = mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("Values\n", values)
print("Attention\n", attention)

Q
 [[-1.57022193  1.36077008 -0.89528795  0.4129017   0.25335323  0.02778555
   1.62836619  0.88931179]
 [-0.58418328  0.5129718  -1.53355145  0.69463447 -1.30660233 -0.06647296
  -0.93841362  2.68778354]
 [-1.64184951  1.92822597 -0.11088405  0.43824264 -0.00870615  0.03803076
   0.42473608 -0.04186264]
 [-2.07997765  2.55894099 -2.28013236  1.4892446  -0.1632833  -0.47578329
   2.11477084 -0.70708352]]
K
 [[ 0.67182034  1.28598954 -1.05083004  0.85078559 -0.99111404 -0.47746124
   0.36717127  0.23699898]
 [-0.71446852  0.04792653  1.41017261  0.99519711 -1.48809583 -1.49773556
   0.99982822 -0.44236456]
 [-0.89614036 -2.0150702  -2.30332924 -0.94571842 -0.68161453  1.31809401
   0.49706184  1.40281078]
 [ 0.91711568  1.5727808  -0.27434931 -0.31215048 -0.58853803 -0.09901833
   1.04709807  1.3507955 ]]
V
 [[ 1.89729554e+00  8.46678539e-01  4.97330176e-01  1.25709236e+00
  -4.21785213e-01  7.19673829e-02 -1.12461151e+00 -3.56175090e-01]
 [-1.99781611e+00 -2.43499718e-01  1.72230363e+0